# GRM shard run (production)

Computes the real sharded GRM (`plink --make-grm-bin --parallel k N_SHARDS`
across all `k`), on this interactive VM, no `dsub`/Batch involved.

Built from `grm_shard_timing.ipynb`'s calibration: per-shard cost is flat
(~810s, ~2% spread) regardless of row range -- fixed cost (loading the full
panel) dominates over the variable relatedness compute -- and peak memory
per shard is small (~0.3 GB) once `--read-freq`/`--memory` are applied, so
concurrency is CPU-bound, not RAM-bound, on this `n1-highmem-64` machine.
At `N_CONCURRENT` shards in flight, wall-clock for `N_SHARDS` in the
200-500 range comes out under ~2 hours -- comfortably an interactive-VM
job, not a cluster one. See that notebook's "Next steps" for why this
doesn't need Batch.

Plink only -- no `GRM-pairs`/`grm_shard_tool` build or code path touched
here; that's a separate later step for the phenotype cross-product.

## Setup

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink" ]; then
  PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
  cd /tmp
  wget -q -O plink.zip "$PLINK_URL"
  unzip -o -q plink.zip plink -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink"
fi

export PATH="$BIN_DIR:$PATH"
plink --version
nproc

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"


## Paths

Same layout as `grm_shard_timing.ipynb` -- local scratch copy of the
genome-wide bed/bim/fam, precomputed allele frequencies. Recomputes
`FREQ_PATH`/`MEMORY_MB` rather than assuming the timing notebook's kernel
is still alive with those variables in memory.

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_grm")
SHARD_OUT_DIR = os.path.join(LOCAL_WORK_DIR, "shards")
os.makedirs(SHARD_OUT_DIR, exist_ok=True)

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = os.path.join(LOCAL_WORK_DIR, BED_NAME)

for ext in ("bed", "bim", "fam"):
    bucket_path = f"{BUCKET_DIR}/{BED_NAME}.{ext}"
    local_path = f"{BED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), f"missing merged bed export: {bucket_path!r} -- run genome_wide_qc_thinning_merge.ipynb's merge section first"
    if not os.path.isfile(local_path):
        import shutil
        shutil.copy(bucket_path, local_path)

print(BED_PREFIX)

In [ ]:
%%bash -s "$BED_PREFIX"
set -e
BED_PREFIX=$1

FREQ_PATH="${BED_PREFIX}_freq.frq"
if [ ! -f "$FREQ_PATH" ]; then
  plink --bfile "$BED_PREFIX" --freq --out "${BED_PREFIX}_freq"
fi

In [ ]:
FREQ_PATH = f"{BED_PREFIX}_freq.frq"
assert os.path.isfile(FREQ_PATH), f"missing precomputed frequencies: {FREQ_PATH!r}"

BED_SIZE_GB = os.path.getsize(f"{BED_PREFIX}.bed") / 1e9
MEMORY_MB = int((BED_SIZE_GB * 2 + 4) * 1024)   # same sizing as grm_shard_timing.ipynb

print(f"FREQ_PATH = {FREQ_PATH}")
print(f"Bed file size: {BED_SIZE_GB:.1f} GB -> --memory {MEMORY_MB} MB")

## Run parameters

`N_SHARDS`: the value settled on from `grm_shard_timing.ipynb`'s
feasibility check -- fill in once that notebook's corrected (CPU-aware)
`max_concurrent`/`wall_clock_hours` numbers look acceptable for the chosen
count. `N_CONCURRENT`: `min(RAM-bound, CPU-bound)` from that same
notebook, not re-derived here -- copy the printed value over rather than
re-measuring, since re-timing shards here would just repeat that
notebook's work.

In [ ]:
N_SHARDS = 300      # per-shard cost is flat (~809s, 2% spread) regardless of N_SHARDS, so
                    # pick the low end of the 200-500 range -- more shards is pure overhead
N_CONCURRENT = 10   # --make-grm-bin isn't actually single-threaded in practice (contrary to
                    # plink's docs) -- at N_CONCURRENT=62 with no --threads cap, 62 processes
                    # each grabbing dozens of threads drove load average past 400 on 64 vCPUs
                    # and zero shards finished in almost an hour. Fewer concurrent processes,
                    # each explicitly capped via --threads below, avoids that oversubscription.
N_THREADS = 6       # explicit per-shard thread cap (was previously unset/auto-detected,
                    # which is what caused the runaway thread counts) -- N_CONCURRENT * N_THREADS
                    # should stay close to vCPU count (64); adjust the split based on real
                    # completed-shard timing once this run has a data point

In [ ]:
%%bash -s "$BED_PREFIX" "$FREQ_PATH" "$MEMORY_MB" "$N_THREADS" "$N_SHARDS" "$SHARD_OUT_DIR"
set -e
BED_PREFIX=$1
FREQ_PATH=$2
MEMORY_MB=$3
N_THREADS=$4
N_SHARDS=$5
SHARD_OUT_DIR=$6

# single-shard timing check at the real N_THREADS setting, before committing to the full
# N_CONCURRENT x N_SHARDS run -- k=1 is as representative as any other k (grm_shard_timing.ipynb
# confirmed per-shard cost is flat regardless of row range)
TIME_LOG=$(mktemp)
TIMEFORMAT='%R %U %S'
{ time plink --bfile "$BED_PREFIX" --make-grm-bin \
  --parallel 1 "$N_SHARDS" \
  --read-freq "$FREQ_PATH" --memory "$MEMORY_MB" \
  --threads "$N_THREADS" \
  --out "${SHARD_OUT_DIR}/grm_shard_1_of_${N_SHARDS}_test"; } 2>> "$TIME_LOG"

# TIMEFORMAT's line is appended last, after any of plink's own stderr output --
# only the final line matters
read -r REAL_SEC USER_SEC SYS_SEC < <(tail -1 "$TIME_LOG")
python3 -c "
real, user = $REAL_SEC, $USER_SEC
ratio = user / real if real else float('nan')
print(f'real={real:.1f}s  user={user:.1f}s  user/real ratio={ratio:.2f}x '
      f'(effective parallelism achieved out of --threads $N_THREADS requested)')
"
rm -f "$TIME_LOG"

## Driver

Same `subprocess` + `ThreadPoolExecutor` pattern as
`genome_wide_qc_thinning_merge.ipynb`'s chromosome driver -- there it was
capped by vCPU count directly since each `plink2` call was itself
multi-threaded; here each `plink --make-grm-bin` shard is single-process
and the cap comes from `grm_shard_timing.ipynb` instead (RAM- or
CPU-bound, whichever binds). One call per `k` in `range(1, N_SHARDS + 1)`,
same `--read-freq`/`--memory` flags as the timing run.

In [ ]:
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_shard(k, n_shards):
    out_prefix = os.path.join(SHARD_OUT_DIR, f"grm_shard_{k}_of_{n_shards}")
    cmd = ["plink", "--bfile", BED_PREFIX, "--make-grm-bin",
           "--parallel", str(k), str(n_shards),
           "--read-freq", FREQ_PATH, "--memory", str(MEMORY_MB),
           "--threads", str(N_THREADS),
           "--out", out_prefix]

    start = time.monotonic()
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    elapsed_sec = time.monotonic() - start

    return {
        "k": k, "elapsed_sec": elapsed_sec, "returncode": proc.returncode,
        "out_prefix": out_prefix,
        "stdout_tail": proc.stdout[-500:] if proc.returncode != 0 else None,
    }

results = []
start_all = time.monotonic()
with ThreadPoolExecutor(max_workers=N_CONCURRENT) as pool:
    futures = {pool.submit(run_shard, k, N_SHARDS): k for k in range(1, N_SHARDS + 1)}
    for i, future in enumerate(as_completed(futures), 1):
        r = future.result()
        results.append(r)
        status = "ok" if r["returncode"] == 0 else f"FAILED (see stdout_tail)"
        print(f"[{i}/{N_SHARDS}] k={r['k']:>4}: {r['elapsed_sec']:.1f}s, {status}")

wall_clock_hours = (time.monotonic() - start_all) / 3600
print(f"\nAll shards done in {wall_clock_hours:.2f} hours wall-clock")

failed = [r for r in results if r["returncode"] != 0]
if failed:
    print(f"\n{len(failed)} shard(s) FAILED:")
    for r in failed:
        print(f"  k={r['k']}: {r['stdout_tail']}")

## Persist to bucket

Only one `.grm.id` is kept -- every shard sees the identical sample panel
(no per-shard `--keep`), so every shard's `.grm.id` is byte-identical.
Keeping all `N_SHARDS` copies would just be `N_SHARDS - 1` redundant
copies of the same small file; one canonical copy (from `k=1`) is enough
for `grm_shard_tool`'s downstream ID lookup.

In [ ]:
%%bash -s "$SHARD_OUT_DIR" "$BUCKET_DIR" "$N_SHARDS"
set -e
SHARD_OUT_DIR=$1
BUCKET_DIR=$2
N_SHARDS=$3

DEST="${BUCKET_DIR}/grm_shards"
mkdir -p "$DEST"

for k in $(seq 1 "$N_SHARDS"); do
  cp "${SHARD_OUT_DIR}/grm_shard_${k}_of_${N_SHARDS}.grm.bin" "${DEST}/"
done

# one canonical .grm.id, from shard k=1 -- every shard's is identical
cp "${SHARD_OUT_DIR}/grm_shard_1_of_${N_SHARDS}.grm.id" "${DEST}/genome_wide.grm.id"

echo "Persisted $(ls "${DEST}"/*.grm.bin | wc -l) shard .grm.bin files + 1 .grm.id to ${DEST}"

## Next steps

Verify before trusting this run:

1. **Total `.grm.bin` size** should equal `N(N+1)/2 x 4 bytes` (`N` =
   sample count) regardless of `N_SHARDS` -- sum the persisted shard file
   sizes and check against that, as a sanity check that no shard silently
   failed or double-counted rows.
2. **`FID` in the one kept `.grm.id`**: `grm_shard_tool`'s pheno lookup
   keys on the full `(FID, IID)` pair, while `write_grm_pheno()` sets
   `FID = IID = person_id` -- plink's `.grm.id` commonly defaults `FID` to
   `"0"`. Check this file's actual `FID` column before the phenotype
   cross-product step trusts it (see `03_grm_shards/README.md`).

`GRM-pairs`/`grm_shard_tool` (`accumulate`/`merge`) is the next stage after
this, and is out of scope here -- separate step, separate notebook, per
the standing note not to touch that submodule as part of shard
construction.